# Notebook for demonstrating and testing MAS capabilities of datasets' creation 

The use of individual workflows of this system is also available to users and developers. 

This journal will demonstrate how to use workflows to generate metadata tables and creating new datasets in datalake.

Also here we will test the capabilities of the presented workflow in terms of the metadata properties creation.

In [ ]:
import sys
sys.path.append('./../src')

In [ ]:
from dotenv import load_dotenv
import json
import os
import shutil

from datalake_driver.drivers.local_storage_driver import LocalDriver
from data_model.datalake import Datalake

load_dotenv('./../src/.env')

In [ ]:
with open('raw_testing_material/imgs_metadata_rich_descr.json', 'r') as i_stream:
    some_dict = json.load(i_stream)

if os.path.exists('./testing_datalake'):
    shutil.rmtree('./testing_datalake')


In [ ]:
local_config = {
    'root_folder': './'
}

driver = LocalDriver(**local_config)

name = 'testing_datalake'
local_datalake = Datalake(name, name + '/', driver)

## Making agents for working with datasets

The function for async testing of dataset creation presented below. It takes as an input some testing parameters (system propmpt, additional info, user's task) and generate statistics with metadata generation result. After that, provided schema can be used to create new dataset.

In [ ]:
import concurrent.futures
from time import time

from mas_exec.scidatamas.dataset_creating_workflow import (
    MDGenerationFlow, MDGenerationState, MetadataStructure)
from utils import TokenUsageCallbackCounter


def generate_schema(user_task: str, system_prompt: str, datalake: Datalake, model: str, provider: str, additional_examples=None, relevant_docs_with_instructions=None):
    creating_datasets_workflow = MDGenerationFlow(datalake=datalake, system_prompt=system_prompt, model=model, provider=provider, is_auto=True, additional_examples=additional_examples, relevant_docs_with_instructions=relevant_docs_with_instructions)

    timeout = 300

    working_state = MDGenerationState()
    working_state['users_task'] = user_task
    wf = creating_datasets_workflow.get_workflow()
    wf = wf.compile()

    errs_cnt = 0
    while errs_cnt <= 5:
        start = time()
        with concurrent.futures.ThreadPoolExecutor() as executor:
            handler = TokenUsageCallbackCounter()
            config = {"callbacks":[handler]}
            future = executor.submit(wf.invoke, working_state, config=config)
            try:
                working_state = future.result(timeout=timeout)
            except concurrent.futures.TimeoutError:
                print(f"Function timed out after {timeout} seconds")
                errs_cnt += 1
                continue
            except Exception as exp:
                print(f"Error: {str(exp)}. Going to restart")
                errs_cnt += 1
                continue
        timing = float(time() - start)
        break

    if errs_cnt > 5:
        timing = None
            
    total_tokens = handler.total_tokens
    return working_state['generation'], total_tokens, timing, errs_cnt

In [ ]:
def count_statistics(schema: MetadataStructure):
    fields_count = len(schema.keys())

    not_str_and_obj_count = 0
    for key, elem in schema.items(): 
        if elem['type'] != 'string' and elem['type'] != 'object': 
            not_str_and_obj_count += 1

    return [fields_count, not_str_and_obj_count]

In [ ]:
import yaml
import os

with open(os.path.join('../src/prompts_templates/creating_md', 'chrono_creating_dataset_sp.yaml')) as stream:
    chrono_prompt = yaml.safe_load(stream)['system_prompt']

chrono_prompt

In [ ]:
task_name = """Hi. 
I want to create a dataset with a metadata table for data that records neuronal activity in the hippocampus during mouse activity. 
The camera is embedded in the live mice' heads and records the neuronal activity during thier activity (like run, sitting, grooming). 
The data is aimed at investigating the effect of neurodegenerative diseases on mouse behavior and signals in the hippocampus."""

In [ ]:
chrono_generation, total_tokens, timing, errs_cnt = generate_schema(task_name, chrono_prompt, local_datalake, model='mistral-large-latest', provider='mistralai')
print(chrono_generation.schema)
chrono_statistics = count_statistics(chrono_generation.schema)
print(f'Chrono sp - Total fields: {chrono_statistics[0]}; Non-string & non-object fields: {chrono_statistics[1]}', total_tokens, timing)

### Checking different prompts (for supplementary).

This experiment proposed to capture difference in built schemas with different system prompts: with instruction with our vision of building metadata and without that vision. As you can see, the counts of different fields are increasing, also increasing non-string types. 

In [ ]:
import yaml
import os

prompts_names = ['common_creating_dataset_sp.yaml', 'chrono_creating_dataset_sp.yaml']

with open(os.path.join('../src/prompts_templates/creating_md', prompts_names[0])) as stream:
    common_prompt = yaml.safe_load(stream)['system_prompt']
with open(os.path.join('../src/prompts_templates/creating_md', prompts_names[1])) as stream:
    chrono_prompt = yaml.safe_load(stream)['system_prompt']

chrono_prompt, common_prompt

In [ ]:
task_name = """Hi. 
I want to create a dataset with a metadata table for data that records neuronal activity in the hippocampus during mouse activity. 
The camera is embedded in the live mice' heads and records the neuronal activity during thier activity (like run, sitting, grooming). 
The data is aimed at investigating the effect of neurodegenerative diseases on mouse behavior and signals in the hippocampus."""

In [ ]:
common_generation, total_tokens, timing, errs_cnt = generate_schema(task_name, common_prompt, local_datalake, model='mistral-large-latest', provider='mistralai')
print(common_generation.schema)
common_statistics = count_statistics(common_generation.schema)
print(f'Common sp - Total fields: {common_statistics[0]}; Non-string & non-object fields: {common_statistics[1]}', total_tokens, timing)

In [ ]:
chrono_generation, total_tokens, timing, errs_cnt = generate_schema(task_name, chrono_prompt, local_datalake, model='mistral-large-latest', provider='mistralai')
print(chrono_generation.schema)
chrono_statistics = count_statistics(chrono_generation.schema)
print(f'Chrono sp - Total fields: {chrono_statistics[0]}; Non-string & non-object fields: {chrono_statistics[1]}', total_tokens, timing)

### Checking average fields generation using 'chrono_generation', 'common_generation' and 'rich_chrono_generation' generations

In [ ]:
MODELS = {'mistralai':'mistral-large-latest'}
PROMPTS = {'common':common_prompt, 'chrono':chrono_prompt}
LAUNCHES_CNT = 16
TASK = """Hi. 
I want to create a dataset with a metadata table for data that records neuronal activity in the hippocampus during mouse activity. 
The camera is embedded in the live mice' heads and records the neuronal activity during thier activity (like run, sitting, grooming). 
The data is aimed at investigating the effect of neurodegenerative diseases on mouse behavior and signals in the hippocampus."""

results = {}

for model_name, model in MODELS.items():
    model_stats = {}
    for prompt_name, prompt in PROMPTS.items():
        stats = []
        
        for i in range(LAUNCHES_CNT):
            print(f'Currently runs: \'{model}\' model, \'{prompt_name}\' sp, launch #{i+1}...')
            gen_res, tokens, cur_time, errs_cnt = generate_schema(TASK, prompt, local_datalake, model=model, provider=model_name)
            new_stat = count_statistics(gen_res.schema)
            total_fields, non_str_fields = new_stat[0], new_stat[1]
            print(f'Total fields: {total_fields}, non-str fields: {non_str_fields}, tokens: {tokens}, time: {cur_time}, errors during execution: {errs_cnt}.')            
            stats.append((total_fields, non_str_fields, tokens, cur_time, errs_cnt))
        
        model_stats.update({prompt_name: stats})
        
    results.update({model_name: model_stats})
        

In [ ]:
import numpy as np

for model_name, model_stats in results.items():
    for prompt_name, stats in model_stats.items():
        total_fields = [run_stat[0] for run_stat in stats]
        non_str_fields = [run_stat[1] for run_stat in stats]
        tokens = [run_stat[2] for run_stat in stats]
        times = [run_stat[3] for run_stat in stats]
        
        mean_fields, std_fields = np.mean(total_fields), np.std(total_fields)
        mean_non_str_fields, std_non_str_fields = np.mean(non_str_fields), np.std(non_str_fields)
        mean_tokens, std_tokens = np.mean(tokens), np.std(tokens)
        mean_times, std_times = np.mean(times), np.std(times)

        print(f"--- '{model_name}'/'{prompt_name}' INFO ---")
        print(f"TOTAL FIELDS:   mean={mean_fields:6.6},\tstd={std_fields:6.6}")
        print(f"NON STR FIELDS: mean={mean_non_str_fields:6.6},\tstd={std_non_str_fields:6.6}")
        print(f"TOKENS:         mean={mean_tokens:6.6},\tstd={std_tokens:6.6}")
        print(f"TIME:           mean={mean_times:6.6},\tstd={std_times:6.6}")
        print()